## GRPO Training — trading_rl

Run this end-to-end on Colab to start training a strategy-optimizing LoRA adapter.

1. **Choose a GPU first** (top-right → *Runtime* → *Change runtime type*: T4 is enough to check the plumbing; A100/L4 for a real run).
2. **Add secrets** (bottom-right sidebar) — `hf_token`, `alpaca_key`, `alpaca_secret`.
3. **Pick `NUM_STEPS`** — start with `1` to verify it loads, then increase.
4. Run every cell top to bottom.

> `trading_rl` is installed from the repo, so **push any local changes first** or local edits won't be visible.

> MLX = local Apple-Silicon; this repo's MLX path is inference-only, so training needs CUDA/Colab. See the notes below.

In [ ]:
# Install trading_rl (+ [train] extra: unsloth, trl, datasets, peft) + vLLM (vLLM makes GRPO generation on CUDA several times faster)
%pip install -q "trading-rl[train] @ git+https://github.com/adhamhelmy/llm-fine-tuning.git" unsloth "trl==0.22.2" datasets peft vllm

In [ ]:
# Shared backtest / prompt / reward code: the `trading_rl` package in this repo
import os, sys
if os.path.isdir('../trading_rl'):   # local clone: import straight from the repo
    sys.path.insert(0, os.path.abspath('..'))
else:                                # Colab / Kaggle
    %pip install -q "trading-rl @ git+https://github.com/adhamhelmy/llm-fine-tuning.git"

### Secrets

Colab Secret manager holds these; the cell below fetches them into `os.environ`. They never touch the notebook.

In [ ]:
import os
from google.colab import userdata

# Colab secret names are case-sensitive: accept either the lowercase name or the env-var name
SECRETS = {'HF_TOKEN': 'hf_token', 'ALPACA_API_KEY': 'alpaca_key', 'ALPACA_SECRET_KEY': 'alpaca_secret'}
for env, alias in SECRETS.items():
    if os.environ.get(env):
        continue
    for name in (env, alias):
        try:
            os.environ[env] = userdata.get(name)
            break
        except Exception as e:
            err = f'{type(e).__name__}: {e}'
    else:
        print(f'Colab secret {env!r} (or {alias!r}) not readable -> {err}')

needed = {k: os.environ.get(k) for k in ('HF_TOKEN', 'ALPACA_API_KEY', 'ALPACA_SECRET_KEY')}
missing = [k for k, v in needed.items() if not v]
print('all secrets present' if not missing else 'MISSING: ' + ', '.join(missing))

### How many steps

GRPO cost scales with `NUM_STEPS`. Start with **`1`** to confirm the model loads, bars cache, and the first reward call works; then raise for a real run.

In [ ]:
import torch
from trading_rl.model import Unsloth        # BEFORE trl / transformers
from trading_rl import (TRAIN_END, TRAIN_START, TRAINING_SYMBOLS,
                        Backtester, RewardConfig, make_strategy_reward, prompt_fn)
from trading_rl.dataset import build_dataset

NUM_STEPS       = 1        # GRPO steps. 1 = quick smoke check; real runs: tens to hundreds
MODEL           = 'unsloth/Ministral-3-3B-Instruct-2512'
FAST_INFERENCE  = True     # vLLM generation during GRPO (needs pip install vllm)

print('CUDA?', torch.cuda.is_available())

# Reward for training AND evaluation. RewardConfig() = avg annual return with 'low'
# costs (1 bp commission + 2 bps slippage per side); RewardConfig.original() = frictionless.
REWARD = RewardConfig()

# One prompt per trading symbol, on the training window.
backtester = Backtester(verbose=False)
backtester.load_bars(TRAINING_SYMBOLS, TRAIN_START, TRAIN_END)
dataset = build_dataset(TRAINING_SYMBOLS, TRAIN_START, TRAIN_END,
                        prompt_fn=prompt_fn(None, backtester))

### Step 0 — sanity check (cheap)

Loads the model and runs the first GRPO step. If it succeeds, the plumbing works.

In [ ]:
model = Unsloth(model_name=MODEL, max_seq_length=5120, max_prompt_length=3072,
                fast_inference=FAST_INFERENCE, gpu_memory_utilization=0.6)
trainer = model.train(
    steps=1,
    reward_functions=[make_strategy_reward(backtester, reward=REWARD)],
    dataset=dataset,
)

### Full run

The same as above, for the full step budget. Saves the LoRA adapter to `grpo_saved_lora/`.

In [ ]:
model = Unsloth(model_name=MODEL, max_seq_length=5120, max_prompt_length=3072,
                fast_inference=FAST_INFERENCE, gpu_memory_utilization=0.6)
trainer = model.train(
    steps=NUM_STEPS,
    reward_functions=[make_strategy_reward(backtester, reward=REWARD)],
    dataset=dataset,
    # push to the Hub (add the hf_token secret in the sidebar):
    # hub_model_id='your-hf-username/ornith-grpo-v1',
    # hf_token=os.environ['HF_TOKEN'],
)
model.unload()  # free the GPU after training